In [55]:
from gen_catalyst_design.discrete_space_diffusion import ExponentialScheduler, UniformTransitionsNoiser, AbsorbingStateNoiser
from gen_catalyst_design.discrete_space_diffusion.Dataset import get_dataloaders_from_atoms_list
from ase_ml_models.databases import get_atoms_list_from_db
from ase_ml_models.utilities import get_connectivity, plot_connectivity
from ase.db import connect
import torch
from ase.io import write
from ase.visualize import view


In [56]:
noiser = "Absorbing"
element_pool = ["(X)"]+["Rh", "Cu", "Au", "Pd", "Ni"]
miller_index = "100"
num_copies = 3

scheduler = ExponentialScheduler(
        beta_max=5e-2, 
        beta_min=1e-4,
        time_sample_method="stratified"
    )

noiser = AbsorbingStateNoiser(
            element_pool=element_pool,
            active_site_freezing=0.0,
            label_smoothing=1e-3
        )

noiser.pre_compute_accum_q_matrices(scheduler=scheduler)

ase_db = connect(f"../../databases/bulk_templates/{miller_index}/{miller_index}_templates.db")
template_atoms = get_atoms_list_from_db(ase_db)[0]
template_atoms.symbols = ["Au" for _ in range(len(template_atoms))]
atoms_list = [template_atoms.copy() for _ in range(num_copies)]

cell = template_atoms.get_cell()


train_loader, val_loader = get_dataloaders_from_atoms_list(
    atoms_list=atoms_list,
    element_pool=element_pool,
    batch_size=num_copies,
    train_val_split=0,
    do_initial_shuffling=False,
    do_train_shuffling=False,
    condition_key=None
)



In [ ]:
timesteps = [0, 250, 500, 750, 1000]
tot_atoms_dict = {}
for timestep in timesteps:
    noised_samples = []
    for batch in train_loader:
        batch_copy = batch.clone()
        noiser.noise_batch_x0_xt(batch=batch_copy, time_batch=timestep*torch.ones(size=(batch_copy.num_nodes,), dtype=torch.long))
        for sample_idx in range(num_copies):
            graph = batch_copy.get_example(sample_idx)
            atoms = graph.to_atoms(element_pool)
            noised_samples.append(atoms)
    tot_atoms_dict[timestep] = noised_samples

view(tot_atoms_dict[750][0], radii=100)
#write("noised_750.traj",tot_atoms_dict[750])
#plot_connectivity(atoms=tot_atoms_dict[750][0], connectivity=template_atoms.info["connectivity"])

    

<Popen: returncode: None args: ['/opt/anaconda3/envs/cat_opt/bin/python', '-...>

: 